# KML Challenge 2026S Starter Notebook

이 노트북은 고객의 트랜잭션 데이터를 고객 단위 feature로 변환한 뒤, **AutoGluon**을 사용하여 성별(`gender`)을 예측하는 Starter Notebook입니다.

### 목표
- 거래 단위 데이터를 고객 단위 데이터로 변환
- manual feature: 구매 카테고리, 상품, 계절, 주중/주말 패턴을 feature로 생성
- word2vec feature: 구매 이력을 문장처럼 다뤄 각 상품을 Word2Vec으로 벡터화하고 그 평균을 고객 feature로 사용  # 임베딩 바꾸기
- AutoGluon `TabularPredictor`로 성별 분류 모델 학습

### 데이터
- `train_transactions.csv`: 학습용 고객 트랜잭션 데이터
- `test_transactions.csv`: 평가용 고객 트랜잭션 데이터
- `y_train.csv`: 고객별 성별 데이터 (타겟 컬럼: `gender`)

## 1. 라이브러리 설치 및 불러오기

AutoGluon이 설치되어 있지 않다면 아래 명령을 먼저 실행!

```python
!pip install autogluon.tabular
```

In [7]:
!pip install autogluon.tabular

In [8]:
import pandas as pd
import numpy as np
from autogluon.tabular import TabularPredictor

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)

## 1-1. (Colab) 구글 드라이브 마운트 및 데이터 경로 설정

Colab에서는 로컬 파일을 직접 읽을 수 없으므로, 데이터를 **구글 드라이브**에 올려두고 마운트해서 사용합니다.

### 사전 준비 (한 번만)
1. 내 구글 드라이브(`MyDrive`)에 **`ML_Final_Competition`** 폴더를 만듭니다.
2. 그 폴더 안에 아래 3개 파일을 업로드합니다.
   - `train_transactions.csv`
   - `test_transactions.csv`
   - `y_train.csv`

아래 셀을 실행하면 드라이브를 마운트하고, 파일이 제대로 있는지 확인합니다.
폴더 이름을 바꿨다면 `DATA_DIR`만 수정하면 됩니다.

In [ ]:
import os

# Colab 환경이면 구글 드라이브 마운트, 아니면 로컬 폴더 사용
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = '/content/drive/MyDrive/ML_Final_Competition'   # 드라이브 내 데이터 폴더
except ImportError:
    DATA_DIR = '.'   # 로컬(주피터 등)에서는 현재 폴더

# 결과 저장 위치(드라이브에 바로 저장하면 런타임이 끊겨도 보존됨)
OUTPUT_DIR = DATA_DIR

# 데이터 파일이 모두 있는지 확인
required = ['train_transactions.csv', 'test_transactions.csv', 'y_train.csv']
print('DATA_DIR =', DATA_DIR)
for f in required:
    path = os.path.join(DATA_DIR, f)
    status = 'OK' if os.path.exists(path) else '없음 (업로드 필요!)'
    print(f'  - {f}: {status}')

## 2. 데이터 불러오기

- `X_train`은 거래 단위 데이터이기 때문에 한 고객이 여러 행에 반복해서 등장할 수 있음.
- 성별 예측을 위해서는 거래 단위 데이터를 **고객 단위 데이터**로 집계해야 함.

In [ ]:
X_train_trans = pd.read_csv(os.path.join(DATA_DIR, 'train_transactions.csv'), encoding='cp949')
y_train = pd.read_csv(os.path.join(DATA_DIR, 'y_train.csv'))

display(X_train_trans.head())
display(y_train.head())

## 3. Manual Feature
- 아래 함수는 백화점 거래 단위 데이터를 고객(`custid`) 단위 feature 테이블로 변환
- 생성하는 feature는 다음과 같음:

1. **기본 구매량 / 다양성**: 전체 거래 수, 고유 상품·브랜드·코너·파트·팀·점포 수, 방문일수, 상품/브랜드 다양성, 방문당 거래 수
2. **금액 feature**: 총·평균·최대·최소·표준편차·중앙값 순구매액(net_amt), 총구매액 대비 할인율, 방문당 구매액 — 객단가와 소비 규모를 반영
3. **행동 비율 feature**: 환불 비율, 수입품 구매 비율, 할부 사용 비율 및 평균 할부개월, 주말 구매 비율 — 소비 성향과 생활 패턴
4. **카테고리별 구매 비중**: 대분류(team_nm)·중분류(part_nm)별 구매 비중 — 어떤 상품군을 선호하는지 (성별 신호의 핵심)
5. **점포 · 계절 · 시간대별 구매 비중**: 어느 점포에서, 어느 계절·시간대에 주로 구매하는지
6. **집중도 feature**: 가장 많이 구매한 팀·파트·계절의 비중 — 구매가 특정 상품군이나 시점에 몰려 있는지를 나타냄

In [ ]:
def make_customer_features(df):
    """백화점 거래 데이터를 고객(custid) 단위 feature로 변환."""
    df = df.copy()

    # ---- 날짜/시간 파생 (sales_datetime 하나에서 추출) ----
    df['sales_datetime'] = pd.to_datetime(df['sales_datetime'])
    df['sales_day'] = df['sales_datetime'].dt.normalize()   # 날짜만 (방문일수 계산용)
    df['month'] = df['sales_datetime'].dt.month
    df['weekday'] = df['sales_datetime'].dt.weekday
    df['is_weekend'] = df['weekday'].isin([5, 6]).astype(int)
    season_map = {3:'봄',4:'봄',5:'봄',6:'여름',7:'여름',8:'여름',
                  9:'가을',10:'가을',11:'가을',12:'겨울',1:'겨울',2:'겨울'}
    df['season'] = df['month'].map(season_map)
    df['hour'] = df['sales_datetime'].dt.hour
    df['time_zone'] = pd.cut(df['hour'], bins=[-1,11,14,17,24],
                             labels=['오전','점심','오후','저녁'])
    df['is_refund'] = (df['net_amt'] < 0).astype(int)
    df['is_installment'] = (df['inst_mon'] > 1).astype(int)

    g = df.groupby('custid')

    # ---- 1) 기본 구매량 / 다양성 ----
    base = g.agg(
        total_txn_cnt=('goodcd','size'),
        unique_goods=('goodcd','nunique'),
        unique_brand=('brd_nm','nunique'),
        unique_corner=('corner_nm','nunique'),
        unique_pc=('pc_nm','nunique'),
        unique_part=('part_nm','nunique'),
        unique_team=('team_nm','nunique'),
        unique_store=('str_nm','nunique'),
        visit_days=('sales_day','nunique'),
    )
    base['goods_diversity'] = base['unique_goods'] / base['total_txn_cnt']
    base['brand_diversity'] = base['unique_brand'] / base['total_txn_cnt']
    base['txn_per_visit']  = base['total_txn_cnt'] / base['visit_days']

    # ---- 2) 금액 ----
    amt = g.agg(
        total_net=('net_amt','sum'),
        mean_net=('net_amt','mean'),
        max_net=('net_amt','max'),
        min_net=('net_amt','min'),
        std_net=('net_amt','std'),
        median_net=('net_amt','median'),
        total_tot=('tot_amt','sum'),
        total_dis=('dis_amt','sum'),
    )
    amt['discount_rate'] = amt['total_dis'] / (amt['total_tot'].abs() + 1)
    amt['amt_per_visit'] = amt['total_net'] / base['visit_days']

    # ---- 3) 행동 비율 ----
    behavior = g.agg(
        refund_ratio=('is_refund','mean'),
        import_ratio=('import_flg','mean'),
        installment_ratio=('is_installment','mean'),
        mean_inst_mon=('inst_mon','mean'),
        weekend_ratio=('is_weekend','mean'),
    )

    # ---- 4) 카테고리 비중 (카디널리티 낮은 것만) ----
    team_ratio   = pd.crosstab(df['custid'], df['team_nm'],  normalize='index').add_prefix('ratio_team_')
    part_ratio   = pd.crosstab(df['custid'], df['part_nm'],  normalize='index').add_prefix('ratio_part_')
    season_ratio = pd.crosstab(df['custid'], df['season'],   normalize='index').add_prefix('ratio_season_')
    time_ratio   = pd.crosstab(df['custid'], df['time_zone'],normalize='index').add_prefix('ratio_time_')
    store_ratio  = pd.crosstab(df['custid'], df['str_nm'],   normalize='index').add_prefix('ratio_store_')

    # ---- 5) 집중도 ----
    conc = pd.DataFrame(index=base.index)
    conc['max_team_ratio']   = team_ratio.max(axis=1)
    conc['max_part_ratio']   = part_ratio.max(axis=1)
    conc['max_season_ratio'] = season_ratio.max(axis=1)

    features = pd.concat([base, amt, behavior, team_ratio, part_ratio,
                          season_ratio, time_ratio, store_ratio, conc], axis=1)
    
    return features.fillna(0).reset_index()

## 4. Word2Vec Feature
- 한 고객이 구매한 `corner_nm`(코너)들을 **시간순으로 나열** → 하나의 "문장"
- 각 `corner_nm`을 "단어"로 보고 **Word2Vec** 학습 → 코너별 임베딩 벡터
- 고객이 구매한 코너 벡터들의 **평균** = 그 고객의 feature

In [ ]:
from gensim.models import Word2Vec

TOKEN_COL = 'corner_nm'   # 'brd_nm', 'pc_nm' 등으로 변경 가능
VECTOR_SIZE = 64; WINDOW = 5; MIN_COUNT = 1; SG = 1; EPOCHS = 10; SEED = 42

def train_token_w2v(df, token_col=TOKEN_COL):
    df = df.copy()
    df['sales_datetime'] = pd.to_datetime(df['sales_datetime'])
    df = df.sort_values(['custid','sales_datetime'])
    df[token_col] = df[token_col].astype(str)
    sentences = df.groupby('custid')[token_col].apply(list).tolist()
    return Word2Vec(sentences=sentences, vector_size=VECTOR_SIZE, window=WINDOW,
                    min_count=MIN_COUNT, sg=SG, workers=4, epochs=EPOCHS, seed=SEED)

def make_w2v_features(df, model, token_col=TOKEN_COL):
    df = df.copy()
    df['sales_datetime'] = pd.to_datetime(df['sales_datetime'])
    df = df.sort_values(['custid','sales_datetime'])
    df[token_col] = df[token_col].astype(str)
    vs = model.vector_size
    def avg_vec(tokens):
        vecs = [model.wv[t] for t in tokens if t in model.wv.key_to_index]
        return np.mean(vecs, axis=0) if vecs else np.zeros(vs)
    seq = df.groupby('custid')[token_col].apply(list)
    mat = np.vstack(seq.apply(avg_vec).values)
    cols = [f'{token_col}_w2v_{i}' for i in range(vs)]
    return pd.DataFrame(mat, columns=cols, index=seq.index).reset_index()

## 5. Feature 생성 및 결합
- manual feature + word2vec feature를 `custid` 기준으로 merge

In [ ]:
# 수동 feature
manual_train = make_customer_features(X_train_trans)

# word2vec feature (train으로 학습 후 적용)
w2v_model = train_token_w2v(X_train_trans, token_col=TOKEN_COL)
w2v_train = make_w2v_features(X_train_trans, w2v_model, token_col=TOKEN_COL)

# 결합
X_train = manual_train.merge(w2v_train, on='custid', how='inner')
print('수동:', manual_train.shape[1]-1, '+ w2v:', w2v_train.shape[1]-1,
      '= 결합:', X_train.shape[1]-1, 'features')

train_data = X_train.merge(y_train[['custid','gender']], on='custid', how='inner').drop(columns=['custid'])
print('train_data:', train_data.shape)

## 6. AutoGluon 모델 학습

- AutoGluon은 여러 머신러닝 모델을 자동으로 학습하고 검증하여 성능이 좋은 모델을 선택
- 성별 예측은 이진 분류 문제이므로 `problem_type='binary'`로 지정하고, 평가척도를 `val_metric='roc_auc'`로 지정
- 타겟 데이터 결합: AutoGluon은 하나의 DataFrame 안에 feature와 target 컬럼이 함께 있어야 함.

In [ ]:
# 타겟 데이터 결합
train_data = X_train.merge(
    y_train[['custid', 'gender']],
    on='custid',
    how='inner'
)

# customer_id는 식별자이므로 모델 feature에서 제외
train_data = train_data.drop(columns=['custid'])

print(train_data.shape)
display(train_data.head())
print(train_data['gender'].value_counts() / len(train_data))

In [ ]:
predictor = TabularPredictor(
    label='gender',
    problem_type='binary',
    eval_metric='roc_auc',
    verbosity=0
).fit(
    train_data=train_data,
    presets='good_quality',
    time_limit=300
)

leaderboard = predictor.leaderboard(train_data, silent=True)
display(leaderboard)

## 7. 테스트 데이터 예측 및 제출 파일 생성
- 테스트 데이터도 학습 데이터와 동일한 방식으로 feature를 생성

In [ ]:
X_test_trans = pd.read_csv(os.path.join(DATA_DIR, "test_transactions.csv"), encoding="cp949")

manual_test = make_customer_features(X_test_trans)
w2v_test = make_w2v_features(X_test_trans, w2v_model, token_col=TOKEN_COL)
X_test = manual_test.merge(w2v_test, on='custid', how='inner')

test_customer_id = X_test["custid"]
X_test = X_test.drop(columns=["custid"])

# 예측 확률
pred_proba = predictor.predict_proba(X_test)

# positive class 확률만 제출
submission = pd.DataFrame({
    "custid": test_customer_id,
    "gender": pred_proba[predictor.positive_class]
})

submission.to_csv(os.path.join(OUTPUT_DIR, "submission_starter.csv"), index=False)
display(submission.head())

# End